# Temporal Framing & Dataset Splits

This notebook documents the temporal structure of the UCI Credit Card Default dataset and defines the observation window, outcome window, and time-based train/validation/test splits.

**Key constraint**: This dataset lacks true origination timestamps. We must use proxy ordering and explicitly document assumptions.

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset exactly as-is
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(list(df.columns))

## Schema Inspection

### Temporal/Sequential Columns

The dataset contains three groups of time-sequenced features:

1. **Payment Status (PAY_0 to PAY_6)**: Repayment status for months -1 to -6
   - PAY_0: Most recent month (month -1)
   - PAY_2 to PAY_6: Months -2 through -6 (historical)
   - Values: -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8
   - Interpretation: -2 = no consumption, -1 = pay duly, 0 = use of revolving credit, 1+ = delay in months

2. **Bill Amounts (BILL_AMT1 to BILL_AMT6)**: Statement bill amounts
   - BILL_AMT1: Most recent month (month -1)
   - BILL_AMT2 to BILL_AMT6: Months -2 through -6

3. **Payment Amounts (PAY_AMT1 to PAY_AMT6)**: Previous payment amounts
   - PAY_AMT1: Most recent month (month -1)
   - PAY_AMT2 to PAY_AMT6: Months -2 through -6

### Static Features

- **ID**: Unique identifier (1 to 30000)
- **LIMIT_BAL**: Credit limit
- **SEX**: Gender (1=male, 2=female)
- **EDUCATION**: Education level (1-4)
- **MARRIAGE**: Marital status (1-3)
- **AGE**: Age in years

### Target Variable

- **default payment next month**: Binary target (0=no default, 1=default)
  - This is the outcome for month 0 (the month following the most recent observation)

In [ ]:
# Inspect temporal columns
temporal_cols = {
    'payment_status': [f'PAY_{i}' if i > 0 else 'PAY_0' for i in [0, 2, 3, 4, 5, 6]],
    'bill_amounts': [f'BILL_AMT{i}' for i in range(1, 7)],
    'payment_amounts': [f'PAY_AMT{i}' for i in range(1, 7)]
}

print("Temporal column groups:")
for group_name, cols in temporal_cols.items():
    print(f"\n{group_name.upper()}:")
    print(f"  Columns: {cols}")
    if group_name == 'payment_status':
        print(f"  Unique values in PAY_0: {sorted(df['PAY_0'].unique())}")

## Temporal Framing

### Observation Window (Information Available at Decision Time)

**Definition**: The time period for which we have historical information when making a credit decision.

**For this dataset**:
- **Months -6 to -1**: Historical payment behavior, bill amounts, and payment amounts
  - PAY_6, BILL_AMT6, PAY_AMT6: Month -6 (oldest)
  - PAY_5, BILL_AMT5, PAY_AMT5: Month -5
  - PAY_4, BILL_AMT4, PAY_AMT4: Month -4
  - PAY_3, BILL_AMT3, PAY_AMT3: Month -3
  - PAY_2, BILL_AMT2, PAY_AMT2: Month -2
  - PAY_0, BILL_AMT1, PAY_AMT1: Month -1 (most recent, before decision)

**Static features** (available at origination):
- LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE

**Decision point**: End of month -1, using information from months -6 to -1.

### Outcome Window (Default Definition)

**Definition**: The time period over which we observe whether default occurs.

**For this dataset**:
- **Month 0**: The month immediately following the observation window
- **Target**: `default payment next month` = 1 if default occurs in month 0, 0 otherwise

**Default definition**: Based on the dataset documentation, default is defined as failure to make minimum payment for at least one month.

**Limitation**: We only observe a single month outcome. True defaults that occur later (month 1, 2, etc.) are not captured, creating potential label noise from delayed defaults.

## Temporal Split Logic

### Critical Constraint: No True Timestamps

**Problem**: This dataset lacks true origination timestamps or observation dates. We cannot split by actual calendar time.

**Implication**: We must use a proxy for temporal ordering.

### Proxy Ordering: Row Index / ID

**Assumption**: The `ID` column (1 to 30000) or row index approximates temporal order:
- Lower IDs were observed/recorded earlier
- Higher IDs were observed/recorded later

**Rationale**:
- In many credit datasets, records are added sequentially
- ID assignment often correlates with application/observation time
- This is a common assumption when true timestamps are unavailable

**Limitation**: This is an approximation. If IDs were assigned randomly or by other criteria, this proxy fails. However, for this project, we proceed with this assumption and acknowledge it explicitly.

### Split Strategy

**Time-based split using ID as proxy**:

1. **Train set**: IDs 1 to 20,000 (earliest ~67% of data)
2. **Validation set**: IDs 20,001 to 25,000 (middle ~17% of data)
3. **Test set**: IDs 25,001 to 30,000 (latest ~17% of data)

**Rationale**:
- Mimics real-world scenario: train on historical data, validate on recent data, test on most recent data
- Prevents temporal leakage: test set contains "future" observations relative to training
- Exposes model to potential distribution shift over time

In [ ]:
# Define temporal splits using ID as proxy
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

print(f"Train set: {len(train_ids)} samples (IDs {train_ids.min()} to {train_ids.max()})")
print(f"Validation set: {len(val_ids)} samples (IDs {val_ids.min()} to {val_ids.max()})")
print(f"Test set: {len(test_ids)} samples (IDs {test_ids.min()} to {test_ids.max()})")

# Verify no overlap
assert len(set(train_ids) & set(val_ids)) == 0, "Train/Val overlap!"
assert len(set(train_ids) & set(test_ids)) == 0, "Train/Test overlap!"
assert len(set(val_ids) & set(test_ids)) == 0, "Val/Test overlap!"
print("\n✓ No overlap between splits")

## Assumptions & Limitations

### What This Temporal Proxy Captures

1. **Sequential ordering**: Assumes lower IDs correspond to earlier observations
2. **Temporal validation**: Test set represents "future" data relative to training
3. **Distribution shift exposure**: Model performance on later IDs may differ from earlier IDs, exposing temporal degradation
4. **No future leakage**: Strictly prevents using "future" information to predict "past" defaults

### What It Cannot Capture

1. **True calendar time**: We don't know actual observation dates (e.g., were IDs 1-1000 from 2020 or 2023?)
2. **Seasonal effects**: Cannot account for economic cycles, holidays, or seasonal patterns
3. **Macroeconomic shifts**: Cannot identify periods of economic downturn or expansion
4. **Cohort effects**: Cannot distinguish between different origination cohorts if IDs span multiple years
5. **Gap periods**: If there are large gaps in ID assignment, the temporal proxy may be misleading

### Why It Is Still Acceptable

1. **Standard practice**: Using row order/ID as temporal proxy is common in credit modeling when timestamps are unavailable
2. **Conservative approach**: This split is more realistic than random splits, which would create unrealistic optimism
3. **Explicit acknowledgment**: By documenting assumptions, we enable proper interpretation of results
4. **Project scope**: For model development and data-centric analysis, this proxy is sufficient to:
   - Prevent temporal leakage
   - Expose distribution shift
   - Validate model stability over "time"
5. **Complementary to drift detection**: The separate drift detection project can handle true temporal monitoring in production

### Key Takeaway

**This temporal framing is a necessary approximation given data limitations. Results should be interpreted with awareness that true temporal relationships are unknown. The split strategy prioritizes preventing leakage and exposing temporal degradation over perfect temporal alignment.**